# 6. Hyperparameter tuning
Many machine learning models require setting up the configuration of the model to train on data effectively. Poor choice of parameters may lead to insufficient results, and thus many methods have been proposed. Nature Inspired Algorithms (NIA) is a class of approaches that provides suboptimal solutions for hyperparameter tuning in moderate time. Particularly, these methods are metaheuristic search in the search space of parameters.

The following methods will be examined as candidates for tuning:
- Particle Swarm Optimization (PSO)
- Grey Wolf Optimizer (GWO)
- Cuckoo Search Optimizer (CSO)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, MonthLocator, DayLocator, HourLocator, DateFormatter
from typing import Tuple, Dict, List, Union, Any, Callable
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
from xgboost import XGBRegressor
from abc import ABC, abstractmethod
from pyswarms.single import GlobalBestPSO

## Particle Swarm Optimization (PSO)

In [485]:
class Tuner(ABC):
    """
    Abstract class for model tuning.

    Attributes:
        model (ForecastModel): model to be tuned.
    """

    model: ForecastModel

    def __init__(self, model: ForecastModel):
        self.model = model

    @abstractmethod
    def tune(self) -> Dict[str, Any]:
        """
        Finds the suboptimal combination of hyperparameters.

        Returns:
            Dict[str, Any]: dictionary of hyperparameter names and values.
        """
        pass

In [509]:
class TuneXGBoost(Tuner):
    """PSO optimization for XGBoost hyperparameters"""
    df: pd.DataFrame

    n_particles: int = 5
    iters: int = 10

    bounds = (
        np.array([0.01, 3, 0.1, 0.1, 0.1, 0]),  # min values
        np.array([0.3, 10, 10, 1, 1, 5])        # max values
    )

    K = 20
    test_size: int = 24*10

    @property
    def metric(self):
        """Protected metric accessor"""
        return mean_squared_error

    def __init__(self, model: ForecastModel, df: pd.DataFrame):
        super().__init__(model)
        self.df = df

    def tune(self):

        def objective_function(params_list):
            scores = []
            for param_set in params_list:
                params = {
                    "learning_rate": param_set[0],
                    "max_depth": int(param_set[1]),
                    "min_child_weight": param_set[2],
                    "subsample": param_set[3],
                    "colsample_bytree": param_set[4],
                    "gamma": param_set[5]
                }
    
                scores.append(self.model.cross_validation(self.df, params, self.K, self.test_size, self.metric))
    
            return np.array(scores)
        
        optimizer = GlobalBestPSO(
            n_particles=self.n_particles,
            dimensions=len(self.bounds[0]),
            options={'c1': 0.5, 'c2': 0.3, 'w': 0.9, 'early_stop': True, 'patience': 3},
            bounds=self.bounds
        )
    
        best_cost, best_params = optimizer.optimize(objective_function, iters=self.iters)
    
        optimized_params = {
            'learning_rate': best_params[0],
            'max_depth': int(best_params[1]),
            'min_child_weight': best_params[2],
            'subsample': best_params[3],
            'colsample_bytree': best_params[4],
            'gamma': best_params[5]
        }
    
        return optimized_params

### Tuning `VolumeFM`

In [510]:
tuner = TuneXGBoost(VolumeFM(), dataset_us_aug)

In [511]:
tuner.tune()

2025-07-01 18:44:07,748 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9, 'early_stop': True, 'patience': 3}
pyswarms.single.global_best: 100%|████████████████████████████████████████████████████████████|10/10, best_cost=6.58e+6
2025-07-01 18:57:14,290 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 6584127.399434492, best pos: [0.03010245 6.26186481 7.24252578 0.51466727 0.67802483 2.84058496]


{'learning_rate': 0.03010244593188737,
 'max_depth': 6,
 'min_child_weight': 7.242525777843713,
 'subsample': 0.5146672731499299,
 'colsample_bytree': 0.6780248304601058,
 'gamma': 2.840584960449063}

### Tuning `ClosePriceFM`

In [512]:
tuner = TuneXGBoost(ClosePriceFM(), dataset_us_aug)

In [513]:
tuner.tune()

2025-07-01 18:57:14,319 - pyswarms.single.global_best - INFO - Optimize for 10 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9, 'early_stop': True, 'patience': 3}
pyswarms.single.global_best: 100%|███████████████████████████████████████████████████████████████|10/10, best_cost=7.67
2025-07-01 19:53:58,915 - pyswarms.single.global_best - INFO - Optimization finished | best cost: 7.667787727222394, best pos: [0.24715276 6.85554156 6.6443031  0.99611552 0.95211569 2.03380207]


{'learning_rate': 0.24715275863366554,
 'max_depth': 6,
 'min_child_weight': 6.6443030994444765,
 'subsample': 0.9961155168453839,
 'colsample_bytree': 0.9521156946502181,
 'gamma': 2.03380206615277}

### Tuning `LSTM`

In [ ]:
class TuneLSTM(Tuner):
    """PSO optimization for LSTM hyperparameters"""
    df: pd.DataFrame

    n_particles: int = 5
    iters: int = 10

    bounds = (
        np.array([0.0001, 16, 16, 16, 0.1]),  # min values
        np.array([0.1, 256, 256, 256, 0.5])        # max values
    )

    K = 20
    test_size: int = 24*10

    @property
    def metric(self):
        """Protected metric accessor"""
        return mean_squared_error

    def __init__(self, model: ForecastModel, df: pd.DataFrame):
        super().__init__(model)
        self.df = df

    def tune(self):

        def objective_function(params_list):
            scores = []
            for param_set in params_list:
                params = {
                    "learning_rate": param_set[0],
                    "layer1": int(param_set[1]),
                    "layer2": int(param_set[2]),
                    "layer3": int(param_set[3]),
                    "dropout": param_set[4]
                }
    
                scores.append(self.model.cross_validation(self.df, params, self.K, self.test_size, self.metric))
    
            return np.array(scores)
        
        optimizer = GlobalBestPSO(
            n_particles=self.n_particles,
            dimensions=len(self.bounds[0]),
            options={'c1': 0.5, 'c2': 0.3, 'w': 0.9, 'early_stop': True, 'patience': 3},
            bounds=self.bounds
        )
    
        best_cost, best_params = optimizer.optimize(objective_function, iters=self.iters)
    
        optimized_params = {
            "learning_rate": param_set[0],
            "layer1": int(param_set[1]),
            "layer2": int(param_set[2]),
            "layer3": int(param_set[3]),
            "dropout": param_set[4]
        }
    
        return optimized_params